# Benchmark de Representações em Visão Computacional — EuroSAT

**CNN do Zero · Transfer Learning · Vision Transformer · LoRA · Embeddings + ML Clássico**

Notebook didático de benchmark entre diferentes estratégias de aprendizado de representação
aplicadas à classificação multiclasse de imagens de satélite (EuroSAT).

In [8]:
from src import (
    Analytics,
    SelecaoFeatures,
    PreProcessamento,
    Modelagem,
)

# <font color='red' style='font-size: 40px;'> Problema de Negócio </font>
<hr style='border: 2px solid red;'>

## Objetivo

Comparar, sob os mesmos critérios experimentais, diferentes estratégias de aprendizado de
representações visuais aplicadas à classificação multiclasse de imagens de satélite do
dataset **EuroSAT**, avaliando não apenas a acurácia, mas o **custo-benefício** de cada
abordagem (custo computacional, interpretabilidade, robustez e capacidade de generalização).

## Dataset

[EuroSAT](https://github.com/phelber/EuroSAT) — imagens Sentinel-2, classificação multiclasse
de uso e ocupação do solo. As imagens estão organizadas em `data/`, uma subpasta por classe
(padrão `ImageFolder` do PyTorch):

```
data/
├── AnnualCrop/
├── Forest/
├── HerbaceousVegetation/
├── Highway/
├── Industrial/
├── Pasture/
├── PermanentCrop/
├── Residential/
├── River/
└── SeaLake/
```

## Tipo de Problema

Classificação de imagens **multiclasse** (10 classes), com uma única classe correta por imagem.

## Motivação do Benchmark

A escolha de arquitetura em produção não deve ser guiada apenas por acurácia. Existe um
**trade-off multidimensional** entre:

- Performance preditiva (Accuracy, F1, AUC)
- Custo computacional de treino e inferência
- Interpretabilidade (relevante inclusive sob ótica de governança)
- Robustez a variações da imagem de entrada
- Velocidade de entrega / facilidade de manutenção

## Critérios de Comparação entre Modelos

| Critério | Descrição |
|---|---|
| Accuracy, Precision, Recall, F1-score, AUC | Qualidade preditiva |
| Tempo de treino | Custo de desenvolvimento |
| Tempo de inferência | Custo operacional em produção |
| Nº de parâmetros (totais e treináveis) | Complexidade / custo de armazenamento |
| Uso de memória | Viabilidade de deploy |
| Tamanho do modelo salvo (MB) | Custo de storage / distribuição |

> **Importante:** o objetivo deste notebook é priorizar simplicidade, ausência de overfitting
> e ausência de vazamento de dados (leakage) em detrimento de ganhos marginais de acurácia
> obtidos à custa de complexidade desnecessária.

# <font color='red' style='font-size: 40px;'> Bibliotecas Utilizadas </font>
<hr style='border: 2px solid red;'>

Utilizamos **PyTorch** como framework principal de Deep Learning (flexibilidade para
customizar arquiteturas e loops de treino), **LightGBM** como modelo clássico de referência
sobre embeddings, **Optuna** para otimização Bayesiana de hiperparâmetros, **peft** para
LoRA, e **scikit-learn / UMAP** para redução de dimensionalidade e métricas.

Fixamos uma seed global (`SEED = 42`) em todas as bibliotecas com componente estocástico,
garantindo reprodutibilidade dos experimentos.

In [ ]:
# =========================================================
# BIBLIOTECAS PADRÃO
# =========================================================
import os
import time
import copy
import random
import warnings
from functools import wraps
from pathlib import Path

warnings.filterwarnings("ignore")

# =========================================================
# MANIPULAÇÃO DE DADOS E VISUALIZAÇÃO
# =========================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# =========================================================
# DEEP LEARNING — PYTORCH
# =========================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

# torchinfo para model summary (instalar se necessário: pip install torchinfo)
try:
    from torchinfo import summary as torch_summary
except ImportError:
    torch_summary = None

# =========================================================
# MODELOS PRÉ-TREINADOS DE VISION TRANSFORMER
# =========================================================
try:
    import timm
except ImportError:
    timm = None

# =========================================================
# LORA (PARAMETER-EFFICIENT FINE-TUNING)
# =========================================================
try:
    from peft import LoraConfig, get_peft_model
except ImportError:
    LoraConfig, get_peft_model = None, None

# =========================================================
# MACHINE LEARNING CLÁSSICO
# =========================================================
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.manifold import TSNE
try:
    import umap
except ImportError:
    umap = None

# =========================================================
# OTIMIZAÇÃO DE HIPERPARÂMETROS
# =========================================================
import optuna
from optuna.visualization import (
    plot_optimization_history, plot_param_importances
)

# =========================================================
# INTERPRETABILIDADE
# =========================================================
import cv2

# =========================================================
# REPRODUTIBILIDADE
# =========================================================
SEED = 42

def set_seed(seed: int = SEED):
    """Fixa a seed em todas as bibliotecas com componente estocástica."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# =========================================================
# DEVICE
# =========================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo utilizado: {DEVICE}")

# =========================================================
# DIRETÓRIO DE DADOS (ver Regra 0 da especificação)
# =========================================================
DATA_DIR = Path("data/")
CLASSES = [
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
    "Pasture", "PermanentCrop", "Residential", "River", "SeaLake"
]
NUM_CLASSES = len(CLASSES)

# <font color='red' style='font-size: 40px;'> Funções Auxiliares </font>
<hr style='border: 2px solid red;'>

Nesta seção implementamos todas as funções reutilizáveis do notebook. A ideia é evitar
duplicação de código: as mesmas funções de treino, avaliação e visualização serão
reutilizadas pela CNN, pela ResNet e pelo ViT.

# <font color='green' style='font-size: 30px;'> Métricas de Classificação </font>
<hr style='border: 2px solid green;'>

In [ ]:
def calcular_metricas(y_true, y_pred, y_proba=None):
    """
    Calcula as principais métricas de classificação multiclasse.

    Parâmetros
    ----------
    y_true : array-like — rótulos verdadeiros
    y_pred : array-like — rótulos preditos (classe com maior probabilidade)
    y_proba : array-like, opcional — probabilidades por classe (necessário para AUC)

    Retorna
    -------
    dict com Accuracy, Precision, Recall, F1-score e AUC (macro, quando aplicável).
    """
    metricas = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

    if y_proba is not None:
        try:
            metricas["auc"] = roc_auc_score(
                y_true, y_proba, multi_class="ovr", average="macro"
            )
        except ValueError:
            metricas["auc"] = np.nan
    else:
        metricas["auc"] = np.nan

    return metricas


def tabela_metricas(dict_splits):
    """
    Recebe um dicionário {"treino": metricas_dict, "validacao": ..., "teste": ...}
    e retorna um DataFrame formatado para exibição.
    """
    df = pd.DataFrame(dict_splits).T
    df = df[["accuracy", "precision", "recall", "f1", "auc"]]
    return df.round(4)

# <font color='green' style='font-size: 30px;'> Visualização de Matriz de Confusão e Curva ROC </font>
<hr style='border: 2px solid green;'>

In [ ]:
def plot_matriz_confusao(y_true, y_pred, classes, titulo="Matriz de Confusão"):
    """Plota a matriz de confusão normalizada por linha (recall por classe)."""
    cm = confusion_matrix(y_true, y_pred, normalize="true")
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=classes, yticklabels=classes)
    plt.title(titulo)
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def plot_curva_roc(y_true, y_proba, classes, titulo="Curva ROC (One-vs-Rest)"):
    """Plota a curva ROC one-vs-rest para cada classe."""
    y_true_bin = pd.get_dummies(y_true).values
    plt.figure(figsize=(8, 6))
    for i, classe in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
        auc_i = roc_auc_score(y_true_bin[:, i], y_proba[:, i])
        plt.plot(fpr, tpr, label=f"{classe} (AUC={auc_i:.2f})")
    plt.plot([0, 1], [0, 1], "k--", label="Aleatório")
    plt.xlabel("Taxa de Falsos Positivos")
    plt.ylabel("Taxa de Verdadeiros Positivos")
    plt.title(titulo)
    plt.legend(fontsize=8, loc="lower right")
    plt.tight_layout()
    plt.show()

# <font color='green' style='font-size: 30px;'> Visualização de Imagens, Filtros e Feature Maps </font>
<hr style='border: 2px solid green;'>

In [ ]:
def visualizar_amostras(dataset, classes, n=9):
    """Exibe um grid n (quadrado) de amostras aleatórias do dataset."""
    lado = int(np.sqrt(n))
    fig, axes = plt.subplots(lado, lado, figsize=(2.2 * lado, 2.2 * lado))
    indices = random.sample(range(len(dataset)), n)
    for ax, idx in zip(axes.flatten(), indices):
        img, label = dataset[idx]
        if isinstance(img, torch.Tensor):
            img = img.permute(1, 2, 0).numpy()
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        ax.imshow(img)
        ax.set_title(classes[label], fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def visualizar_filtros(modelo, nome_camada, n_filtros=16):
    """Visualiza os pesos (filtros) de uma camada convolucional específica."""
    camada = dict(modelo.named_modules())[nome_camada]
    pesos = camada.weight.data.clone().cpu()
    n_filtros = min(n_filtros, pesos.shape[0])
    lado = int(np.ceil(np.sqrt(n_filtros)))

    fig, axes = plt.subplots(lado, lado, figsize=(1.6 * lado, 1.6 * lado))
    for i, ax in enumerate(axes.flatten()):
        if i < n_filtros:
            f = pesos[i]
            f = f.mean(dim=0)  # média entre canais de entrada para visualização
            f = (f - f.min()) / (f.max() - f.min() + 1e-8)
            ax.imshow(f, cmap="viridis")
        ax.axis("off")
    plt.suptitle(f"Filtros da camada '{nome_camada}'")
    plt.tight_layout()
    plt.show()


def visualizar_feature_maps(modelo, nome_camada, imagem_tensor, n_maps=16):
    """Visualiza os feature maps produzidos por uma camada para uma imagem de entrada."""
    ativacoes = {}

    def hook(module, entrada, saida):
        ativacoes["saida"] = saida.detach().cpu()

    camada = dict(modelo.named_modules())[nome_camada]
    handle = camada.register_forward_hook(hook)

    modelo.eval()
    with torch.no_grad():
        modelo(imagem_tensor.unsqueeze(0).to(DEVICE))
    handle.remove()

    maps = ativacoes["saida"][0]
    n_maps = min(n_maps, maps.shape[0])
    lado = int(np.ceil(np.sqrt(n_maps)))

    fig, axes = plt.subplots(lado, lado, figsize=(1.6 * lado, 1.6 * lado))
    for i, ax in enumerate(axes.flatten()):
        if i < n_maps:
            ax.imshow(maps[i], cmap="viridis")
        ax.axis("off")
    plt.suptitle(f"Feature maps da camada '{nome_camada}'")
    plt.tight_layout()
    plt.show()

# <font color='green' style='font-size: 30px;'> Comparação de Modelos, Tempo e Contagem de Parâmetros </font>
<hr style='border: 2px solid green;'>

In [ ]:
def medir_tempo(func):
    """Decorator que mede o tempo de execução de uma função em segundos."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        inicio = time.time()
        resultado = func(*args, **kwargs)
        duracao = time.time() - inicio
        return resultado, duracao
    return wrapper


def contar_parametros(modelo):
    """Retorna (parâmetros totais, parâmetros treináveis) de um modelo PyTorch."""
    total = sum(p.numel() for p in modelo.parameters())
    treinaveis = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    return total, treinaveis


def tamanho_modelo_mb(modelo, caminho_temp="temp_model.pt"):
    """Salva temporariamente o state_dict do modelo e retorna o tamanho em MB."""
    torch.save(modelo.state_dict(), caminho_temp)
    tamanho = os.path.getsize(caminho_temp) / (1024 ** 2)
    os.remove(caminho_temp)
    return round(tamanho, 2)


def comparar_modelos(lista_resultados):
    """
    Recebe uma lista de dicionários com os resultados de cada modelo e retorna
    um DataFrame consolidado, ordenado por F1-score decrescente.
    """
    df = pd.DataFrame(lista_resultados)
    colunas_ordem = [
        "modelo", "accuracy", "precision", "recall", "f1", "auc",
        "tempo_treino_s", "tempo_inferencia_s",
        "parametros_totais", "parametros_treinaveis", "tamanho_mb"
    ]
    colunas_existentes = [c for c in colunas_ordem if c in df.columns]
    df = df[colunas_existentes].sort_values("f1", ascending=False).reset_index(drop=True)
    return df

# <font color='green' style='font-size: 30px;'> Extração de Embeddings </font>
<hr style='border: 2px solid green;'>

In [ ]:
def extrair_embeddings(modelo, dataloader, device=DEVICE):
    """
    Extrai embeddings (saída do backbone, antes da camada de classificação final)
    de todas as imagens de um DataLoader.

    Retorna
    -------
    embeddings : np.ndarray de shape (n_amostras, dim_embedding)
    labels : np.ndarray de shape (n_amostras,)
    """
    modelo.eval()
    todos_embeddings, todos_labels = [], []

    with torch.no_grad():
        for imagens, labels in dataloader:
            imagens = imagens.to(device)
            saida = modelo(imagens)
            if saida.dim() > 2:
                saida = torch.flatten(saida, 1)
            todos_embeddings.append(saida.cpu().numpy())
            todos_labels.append(labels.numpy())

    return np.concatenate(todos_embeddings), np.concatenate(todos_labels)

# <font color='green' style='font-size: 30px;'> Loop de Treinamento Genérico </font>
<hr style='border: 2px solid green;'>

A função abaixo implementa um **loop de treinamento genérico**, reutilizado pela CNN
baseline, pela ResNet50 (Transfer Learning) e pelo ViT (Fine-Tuning / LoRA). O objetivo é
evitar duplicação de código e garantir que todas as arquiteturas sejam treinadas e avaliadas
sob o mesmo protocolo (mesmo critério de parada, mesmo tipo de agendamento de LR, etc.).

Etapas de cada iteração (mini-batch):

1. **Forward:** a imagem passa pela rede e gera os logits de saída
2. **Cálculo da Loss:** comparação entre os logits e o rótulo verdadeiro via Cross-Entropy
3. **Zero Grad:** zera os gradientes acumulados do passo anterior
4. **Backpropagation:** calcula o gradiente da loss em relação a cada peso da rede
5. **Atualização dos Pesos:** o otimizador move os pesos na direção que reduz a loss

In [ ]:
def treinar_modelo(modelo, train_loader, val_loader, criterion, optimizer,
                    scheduler=None, epochs=30, device=DEVICE,
                    early_stopping_patience=5, nome_modelo="modelo"):
    """
    Loop de treinamento genérico com early stopping baseado na loss de validação.

    Retorna
    -------
    modelo : melhor modelo encontrado (pesos restaurados do melhor epoch)
    historico : dict com listas de loss/accuracy/lr por época (treino e validação)
    """
    modelo.to(device)
    historico = {
        "train_loss": [], "val_loss": [],
        "train_acc": [], "val_acc": [],
        "lr": [], "tempo_epoca_s": []
    }

    melhor_val_loss = float("inf")
    melhor_state = copy.deepcopy(modelo.state_dict())
    paciencia_atual = 0

    for epoca in range(epochs):
        inicio_epoca = time.time()

        # ---------------------- TREINO ----------------------
        modelo.train()
        loss_treino, acertos_treino, total_treino = 0.0, 0, 0

        for imagens, labels in train_loader:
            imagens, labels = imagens.to(device), labels.to(device)

            optimizer.zero_grad()                     # (4) zera gradientes
            saida = modelo(imagens)                    # (1) forward
            loss = criterion(saida, labels)             # (2) cálculo da loss
            loss.backward()                              # (3) backpropagation
            optimizer.step()                              # (5) atualização dos pesos

            loss_treino += loss.item() * imagens.size(0)
            acertos_treino += (saida.argmax(1) == labels).sum().item()
            total_treino += imagens.size(0)

        loss_treino /= total_treino
        acc_treino = acertos_treino / total_treino

        # ---------------------- VALIDAÇÃO ----------------------
        modelo.eval()
        loss_val, acertos_val, total_val = 0.0, 0, 0

        with torch.no_grad():
            for imagens, labels in val_loader:
                imagens, labels = imagens.to(device), labels.to(device)
                saida = modelo(imagens)
                loss = criterion(saida, labels)

                loss_val += loss.item() * imagens.size(0)
                acertos_val += (saida.argmax(1) == labels).sum().item()
                total_val += imagens.size(0)

        loss_val /= total_val
        acc_val = acertos_val / total_val

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(loss_val)
            else:
                scheduler.step()

        lr_atual = optimizer.param_groups[0]["lr"]
        tempo_epoca = time.time() - inicio_epoca

        historico["train_loss"].append(loss_treino)
        historico["val_loss"].append(loss_val)
        historico["train_acc"].append(acc_treino)
        historico["val_acc"].append(acc_val)
        historico["lr"].append(lr_atual)
        historico["tempo_epoca_s"].append(tempo_epoca)

        print(f"[{nome_modelo}] Época {epoca+1:02d}/{epochs} | "
              f"train_loss={loss_treino:.4f} val_loss={loss_val:.4f} | "
              f"train_acc={acc_treino:.4f} val_acc={acc_val:.4f} | "
              f"lr={lr_atual:.2e} | {tempo_epoca:.1f}s")

        # ---------------------- EARLY STOPPING ----------------------
        if loss_val < melhor_val_loss:
            melhor_val_loss = loss_val
            melhor_state = copy.deepcopy(modelo.state_dict())
            paciencia_atual = 0
        else:
            paciencia_atual += 1
            if paciencia_atual >= early_stopping_patience:
                print(f"Early stopping na época {epoca+1} "
                      f"(sem melhora por {early_stopping_patience} épocas).")
                break

    modelo.load_state_dict(melhor_state)
    return modelo, historico


def plot_historico_treino(historico, titulo="Histórico de Treinamento"):
    """Plota loss, accuracy, learning rate e tempo por época."""
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))

    axes[0, 0].plot(historico["train_loss"], label="Treino")
    axes[0, 0].plot(historico["val_loss"], label="Validação")
    axes[0, 0].set_title("Loss por Época")
    axes[0, 0].set_xlabel("Época"); axes[0, 0].set_ylabel("Loss"); axes[0, 0].legend()

    axes[0, 1].plot(historico["train_acc"], label="Treino")
    axes[0, 1].plot(historico["val_acc"], label="Validação")
    axes[0, 1].set_title("Accuracy por Época")
    axes[0, 1].set_xlabel("Época"); axes[0, 1].set_ylabel("Accuracy"); axes[0, 1].legend()

    axes[1, 0].plot(historico["lr"], color="darkorange")
    axes[1, 0].set_title("Learning Rate por Época")
    axes[1, 0].set_xlabel("Época"); axes[1, 0].set_ylabel("LR")

    axes[1, 1].plot(historico["tempo_epoca_s"], color="green")
    axes[1, 1].set_title("Tempo por Época (s)")
    axes[1, 1].set_xlabel("Época"); axes[1, 1].set_ylabel("Segundos")

    plt.suptitle(titulo)
    plt.tight_layout()
    plt.show()


@medir_tempo
def avaliar_modelo(modelo, dataloader, device=DEVICE):
    """Avalia um modelo em um DataLoader e retorna y_true, y_pred, y_proba."""
    modelo.eval()
    y_true, y_pred, y_proba = [], [], []

    with torch.no_grad():
        for imagens, labels in dataloader:
            imagens = imagens.to(device)
            saida = modelo(imagens)
            probs = F.softmax(saida, dim=1).cpu().numpy()

            y_true.extend(labels.numpy())
            y_pred.extend(probs.argmax(axis=1))
            y_proba.extend(probs)

    return np.array(y_true), np.array(y_pred), np.array(y_proba)

# <font color='red' style='font-size: 40px;'> Leitura dos Dados </font>
<hr style='border: 2px solid red;'>

## Estrutura de Pastas

O EuroSAT segue o padrão esperado por `torchvision.datasets.ImageFolder`: uma subpasta por
classe dentro de `data/`, cada uma contendo as imagens daquela classe (ver árvore de pastas
na Seção 1).

## Separação Treino / Validação / Teste

Adotamos a proporção **70% / 15% / 15%**, com `random_state=SEED`, aplicada sobre os índices
do dataset completo — garantindo que a mesma imagem não apareça em mais de uma partição
(sem leakage entre conjuntos).

In [ ]:
# =========================================================
# TRANSFORM MÍNIMO PARA LEITURA (o pipeline definitivo é
# construído na Seção 6 — Pré-Processamento)
# =========================================================
transform_leitura = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

dataset_completo = datasets.ImageFolder(root=str(DATA_DIR), transform=transform_leitura)

# Garante que a ordem das classes do ImageFolder bate com CLASSES definida acima
assert dataset_completo.classes == CLASSES, (
    "A ordem de classes do ImageFolder difere de CLASSES — ajuste a lista CLASSES."
)

n_total = len(dataset_completo)
n_treino = int(0.70 * n_total)
n_val = int(0.15 * n_total)
n_teste = n_total - n_treino - n_val

gerador = torch.Generator().manual_seed(SEED)
dataset_treino, dataset_val, dataset_teste = random_split(
    dataset_completo, [n_treino, n_val, n_teste], generator=gerador
)

print(f"Total de imagens: {n_total}")
print(f"Treino:     {len(dataset_treino)} ({len(dataset_treino)/n_total:.1%})")
print(f"Validação:  {len(dataset_val)} ({len(dataset_val)/n_total:.1%})")
print(f"Teste:      {len(dataset_teste)} ({len(dataset_teste)/n_total:.1%})")

# <font color='red' style='font-size: 40px;'> Análise Exploratória das Imagens </font>
<hr style='border: 2px solid red;'>

# <font color='green' style='font-size: 30px;'> Distribuição de Imagens por Classe </font>
<hr style='border: 2px solid green;'>

In [ ]:
labels_completo = [label for _, label in dataset_completo.samples]
contagem_classes = pd.Series(labels_completo).map(
    dict(enumerate(CLASSES))
).value_counts().reindex(CLASSES)

plt.figure(figsize=(10, 5))
sns.barplot(x=contagem_classes.index, y=contagem_classes.values, palette="viridis")
plt.title("Distribuição de Imagens por Classe — EuroSAT")
plt.xlabel("Classe")
plt.ylabel("Nº de Imagens")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

contagem_classes

**Interpretação:** se a distribuição entre classes for razoavelmente equilibrada, não é
necessário aplicar técnicas de balanceamento (oversampling/undersampling ou `class_weight`).
Caso haja desbalanceamento relevante (ex.: razão > 3:1 entre a classe majoritária e a
minoritária), isso deve ser tratado na função de perda (Seção 7) via pesos de classe.

# <font color='green' style='font-size: 30px;'> Resolução das Imagens </font>
<hr style='border: 2px solid green;'>

In [ ]:
amostra_resolucoes = []
for caminho, _ in random.sample(dataset_completo.samples, min(500, n_total)):
    with Image.open(caminho) as img:
        amostra_resolucoes.append(img.size)  # (largura, altura)

df_resolucoes = pd.DataFrame(amostra_resolucoes, columns=["largura", "altura"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_resolucoes["largura"], ax=axes[0], kde=False)
axes[0].set_title("Distribuição de Largura")
sns.histplot(df_resolucoes["altura"], ax=axes[1], kde=False)
axes[1].set_title("Distribuição de Altura")
plt.tight_layout()
plt.show()

df_resolucoes.describe()

# <font color='green' style='font-size: 30px;'> Visualização de Amostras por Classe </font>
<hr style='border: 2px solid green;'>

In [ ]:
visualizar_amostras(dataset_treino, CLASSES, n=9)

# <font color='green' style='font-size: 30px;'> Discussão dos Resultados da EDA </font>
<hr style='border: 2px solid green;'>

- **Balanceamento:** avaliar a tabela de contagem por classe gerada acima — o EuroSAT costuma
  apresentar leve desbalanceamento entre algumas classes (ex. `Highway` e `River` tendem a ter
  menos amostras que `AnnualCrop` ou `SeaLake`), sem, no entanto, configurar um cenário de
  classes raras.
- **Resolução:** as imagens do EuroSAT (versão RGB) são tipicamente uniformes em 64×64 pixels,
  o que simplifica o pipeline de pré-processamento (ainda assim, mantém-se o `Resize` explícito
  por robustez, caso a fonte de dados varie).
- **Qualidade:** não foram identificadas, a priori, imagens corrompidas — a leitura via
  `ImageFolder` já falharia nesse caso. Ainda assim, recomenda-se rodar uma validação de
  integridade em produção antes de treinar.

# <font color='red' style='font-size: 40px;'> Pré-Processamento </font>
<hr style='border: 2px solid red;'>

# <font color='green' style='font-size: 30px;'> Resize </font>
<hr style='border: 2px solid green;'>

Padronizamos todas as imagens para uma resolução fixa de entrada da rede. Isso é necessário
porque camadas totalmente conectadas (e o próprio produto matricial das convoluções) exigem
dimensões de entrada consistentes. Usamos **224×224** para viabilizar o uso direto de
backbones pré-treinados (ResNet50 e ViT foram treinados originalmente nessa resolução).

# <font color='green' style='font-size: 30px;'> Normalização </font>
<hr style='border: 2px solid green;'>

Normalizamos cada canal (R, G, B) subtraindo a média e dividindo pelo desvio-padrão:

$$x_{norm} = \frac{x - \mu}{\sigma}$$

Para os modelos pré-treinados (ResNet, ViT) usamos a média/desvio do **ImageNet**
(`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`), pois foi com essa normalização
que os pesos pré-treinados foram originalmente ajustados. Para a CNN treinada do zero, a
mesma normalização é mantida por consistência experimental entre os modelos comparados.

# <font color='green' style='font-size: 30px;'> Data Augmentation </font>
<hr style='border: 2px solid green;'>

Aplicamos augmentation **apenas no conjunto de treino** (nunca em validação/teste, para não
distorcer a estimativa de performance real). Técnicas utilizadas:

| Técnica | Efeito |
|---|---|
| `RandomHorizontalFlip` | Aumenta a variabilidade sem alterar a semântica (útil pois imagens de satélite não têm uma orientação canônica). Reduz overfitting a orientações específicas. |
| `RandomRotation` | Simula diferentes ângulos de captura do satélite. Reduz overfitting a posições fixas de estruturas na imagem. |
| `ColorJitter` (brilho/contraste) | Simula variações de iluminação/atmosfera entre capturas. Reduz overfitting a condições de iluminação específicas. |
| `RandomResizedCrop` | Simula variações de zoom/enquadramento. Aumenta a robustez a pequenas variações de escala. |

Augmentation em excesso pode causar **underfitting** (a rede não converge porque a tarefa fica
artificialmente difícil); a ausência de augmentation, por outro lado, tende a favorecer
**overfitting** em datasets de tamanho moderado como o EuroSAT. O conjunto acima foi escolhido
por preservar a semântica da cena.

# <font color='green' style='font-size: 30px;'> Pipeline de Transformação e DataLoaders </font>
<hr style='border: 2px solid green;'>

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

MEAN_IMAGENET = [0.485, 0.456, 0.406]
STD_IMAGENET = [0.229, 0.224, 0.225]

transform_treino = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN_IMAGENET, STD_IMAGENET),
])

transform_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN_IMAGENET, STD_IMAGENET),
])

# Reconstrói os datasets com os transforms definitivos, mantendo os MESMOS índices
# de treino/val/teste obtidos na Seção 4 (evita leakage entre partições).
dataset_completo_treino_tf = datasets.ImageFolder(root=str(DATA_DIR), transform=transform_treino)
dataset_completo_eval_tf = datasets.ImageFolder(root=str(DATA_DIR), transform=transform_eval)

dataset_treino_final = torch.utils.data.Subset(dataset_completo_treino_tf, dataset_treino.indices)
dataset_val_final = torch.utils.data.Subset(dataset_completo_eval_tf, dataset_val.indices)
dataset_teste_final = torch.utils.data.Subset(dataset_completo_eval_tf, dataset_teste.indices)

train_loader = DataLoader(dataset_treino_final, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(dataset_val_final, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(dataset_teste_final, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Batches — treino: {len(train_loader)} | validação: {len(val_loader)} | teste: {len(test_loader)}")

# <font color='red' style='font-size: 40px;'> Baseline — CNN Desenvolvida do Zero </font>
<hr style='border: 2px solid red;'>

# <font color='green' style='font-size: 30px;'> Construção da Arquitetura </font>
<hr style='border: 2px solid green;'>

A CNN baseline (`EuroSATCNN`) segue um padrão clássico de blocos
**Conv → BatchNorm → Ativação → Pooling**, repetido três vezes, seguido de camadas densas.

- **Nº de camadas convolucionais (3):** suficiente para capturar desde bordas/texturas
  (camadas iniciais) até padrões mais abstratos (camadas finais), sem o custo computacional de
  arquiteturas mais profundas — adequado ao tamanho moderado do EuroSAT (evita overfitting por
  excesso de capacidade).
- **Nº de filtros (32 → 64 → 128):** crescimento geométrico padrão, que compensa a redução
  espacial (pooling) com aumento da profundidade de representação.
- **Kernel size (3×3):** kernel pequeno, eficiente e amplamente validado na literatura (VGG),
  permite empilhar mais camadas com o mesmo campo receptivo de kernels maiores a um custo
  computacional menor.
- **Padding (1):** preserva a dimensão espacial após a convolução (`same padding` para kernel 3×3
  e stride 1), evitando perda de informação nas bordas.
- **Stride (1):** a redução espacial é feita explicitamente pelo `MaxPool2d`, não pela
  convolução — separa a responsabilidade de "extrair padrões" da de "reduzir resolução".
- **Campo receptivo:** após 3 blocos conv+pool, o campo receptivo efetivo cobre uma região
  suficientemente grande da imagem para capturar estruturas como estradas, quadras agrícolas e
  quarteirões residenciais.

# <font color='green' style='font-size: 30px;'> Função de Ativação </font>
<hr style='border: 2px solid green;'>

Comparação das funções de ativação candidatas:

$$\text{ReLU}(x) = \max(0, x)$$
$$\text{LeakyReLU}(x) = \max(\alpha x, x), \quad \alpha \approx 0.01$$
$$\text{GELU}(x) = x \cdot \Phi(x)$$
$$\text{SiLU}(x) = x \cdot \sigma(x)$$

| Ativação | Vantagem | Desvantagem |
|---|---|---|
| ReLU | Simples, rápida, sparsity útil como regularização | "Neurônios mortos" (gradiente zero para x<0) |
| LeakyReLU | Corrige neurônios mortos | Hiperparâmetro adicional (α) |
| GELU | Suave, usada em Transformers modernos | Mais custosa computacionalmente |
| SiLU (Swish) | Suave, bom desempenho empírico em CNNs profundas | Levemente mais custosa que ReLU |

**Escolha:** `ReLU` para a CNN baseline — é o padrão mais robusto e barato computacionalmente
para arquiteturas rasas como a nossa, e evita adicionar complexidade desnecessária (princípio
de simplicidade adotado neste projeto).

# <font color='green' style='font-size: 30px;'> Pooling </font>
<hr style='border: 2px solid green;'>

| Técnica | Efeito |
|---|---|
| Max Pooling | Preserva a ativação mais forte da região — bom para detectar presença de padrões |
| Average Pooling | Suaviza a região — bom para preservar informação de textura geral |
| Global Average Pooling | Substitui o Flatten+FC por uma média global por canal — reduz drasticamente o nº de parâmetros e o risco de overfitting |

**Escolha:** `Max Pooling` entre os blocos convolucionais (preserva bordas/estruturas mais
nitidamente, importante para classes como `Highway` e `River`), e uma camada final de
`Flatten` (não Global Average Pooling) para manter compatibilidade didática com a inspeção de
feature maps.

# <font color='green' style='font-size: 30px;'> Batch Normalization </font>
<hr style='border: 2px solid green;'>

A Batch Normalization normaliza as ativações de cada mini-batch (média 0, variância 1, com
parâmetros aprendíveis de escala e deslocamento). Isso **estabiliza o treinamento** ao evitar
que a distribuição das ativações mude drasticamente entre camadas (internal covariate shift),
e **acelera a convergência**, permitindo o uso de learning rates mais altos.

# <font color='green' style='font-size: 30px;'> Dropout </font>
<hr style='border: 2px solid green;'>

O Dropout desliga aleatoriamente uma fração dos neurônios durante o treino, forçando a rede a
não depender excessivamente de um subconjunto específico de unidades — uma forma de
regularização que reduz overfitting ao criar um efeito equivalente a um ensemble implícito de
sub-redes.

# <font color='green' style='font-size: 30px;'> Definição da Arquitetura (Código) </font>
<hr style='border: 2px solid green;'>

In [ ]:
# =========================================================
# DEFINIÇÃO DA ARQUITETURA DA CNN BASELINE
# =========================================================

class EuroSATCNN(nn.Module):
    """CNN convolucional simples para classificação do EuroSAT (10 classes)."""

    def __init__(self, num_classes=NUM_CLASSES, dropout1=0.25, dropout2=0.5):
        super(EuroSATCNN, self).__init__()

        # ---------------- BLOCO CONVOLUCIONAL 1 ----------------
        # Entrada: 3 canais RGB | Saída: 32 feature maps
        # Kernel 3x3, stride 1, padding 1 -> preserva dimensão espacial
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        # ---------------- BLOCO CONVOLUCIONAL 2 ----------------
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        # ---------------- BLOCO CONVOLUCIONAL 3 ----------------
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        # ---------------- REGULARIZAÇÃO ----------------
        self.dropout1 = nn.Dropout(dropout1)
        self.dropout2 = nn.Dropout(dropout2)

        # ---------------- CAMADAS DENSAS ----------------
        # Após 3 MaxPool2d(2) sobre entrada 224x224: 224 -> 112 -> 56 -> 28
        # Dimensão do flatten: 128 canais * 28 * 28
        self.flatten_dim = 128 * 28 * 28
        self.fc1 = nn.Linear(self.flatten_dim, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Bloco 1: Conv -> BN -> ReLU -> MaxPool
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, 2)

        # Bloco 2
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, 2)

        # Bloco 3
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.max_pool2d(x, 2)

        # Regularização antes do Flatten
        x = self.dropout1(x)

        # Flatten: tensor 4D -> vetor 1D
        x = torch.flatten(x, 1)

        # Camadas densas
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = F.relu(self.fc2(x))

        # Camada de saída (logits — Cross-Entropy aplica o softmax internamente)
        x = self.fc3(x)
        return x


cnn_baseline = EuroSATCNN(num_classes=NUM_CLASSES).to(DEVICE)
print(cnn_baseline)

# <font color='green' style='font-size: 30px;'> Contagem de Parâmetros e Model Summary </font>
<hr style='border: 2px solid green;'>

In [ ]:
total_params, trainable_params = contar_parametros(cnn_baseline)
print(f"Parâmetros totais: {total_params:,}")
print(f"Parâmetros treináveis: {trainable_params:,}")

if torch_summary is not None:
    torch_summary(cnn_baseline, input_size=(1, 3, IMG_SIZE, IMG_SIZE))

# <font color='green' style='font-size: 30px;'> Função de Perda </font>
<hr style='border: 2px solid green;'>

Para classificação multiclasse com rótulos mutuamente exclusivos, usamos **Cross-Entropy
Loss**, que combina `LogSoftmax` + `Negative Log-Likelihood`:

$$
\mathcal{L}_{CE} = -\sum_{c=1}^{C} y_c \log(\hat{p}_c),
\qquad
\hat{p}_c = \frac{e^{z_c}}{\sum_{k=1}^{C} e^{z_k}}
$$

onde $z_c$ é o logit da classe $c$, $\hat p_c$ é a probabilidade prevista (softmax) e $y_c$ é
1 se $c$ for a classe verdadeira e 0 caso contrário. Minimizar essa loss equivale a maximizar
a probabilidade atribuída à classe correta.

Se a distribuição de classes (Seção 5) apresentar desbalanceamento relevante, usamos o
parâmetro `weight` da `CrossEntropyLoss` com pesos inversamente proporcionais à frequência de
cada classe.

# <font color='green' style='font-size: 30px;'> Otimizador </font>
<hr style='border: 2px solid green;'>

| Otimizador | Característica |
|---|---|
| SGD | Simples, mas sensível à escolha de LR; convergência mais lenta |
| SGD + Momentum | Acelera convergência ao acumular direção consistente do gradiente |
| Adam | Adapta o LR por parâmetro (momentos de 1ª e 2ª ordem); convergência rápida |
| AdamW | Como o Adam, mas com weight decay desacoplado da atualização do gradiente — regularização mais correta teoricamente |

**Escolha:** `AdamW`, por oferecer convergência rápida e estável combinada com regularização
L2 correta (evitando o acoplamento problemático entre LR e weight decay presente no Adam
padrão).

# <font color='green' style='font-size: 30px;'> Learning Rate e Scheduler </font>
<hr style='border: 2px solid green;'>

Iniciamos com `lr=1e-3` (valor padrão robusto para AdamW em CNNs de porte pequeno/médio) e
aplicamos `ReduceLROnPlateau`, que reduz o LR automaticamente quando a loss de validação para
de melhorar — evitando o platô de treinamento sem exigir um agendamento manual fixo (como no
StepLR) e sendo menos agressivo que o Cosine Annealing para um dataset de porte moderado.

# <font color='green' style='font-size: 30px;'> Weight Decay e Early Stopping </font>
<hr style='border: 2px solid green;'>

**Weight Decay (regularização L2):** adiciona uma penalidade proporcional à norma dos pesos à
função de perda, desincentivando pesos com magnitude muito alta:

$$\mathcal{L}_{total} = \mathcal{L}_{CE} + \lambda \sum_i w_i^2$$

**Early Stopping:** interrompe o treinamento se a loss de validação não melhorar por
`patience` épocas consecutivas, restaurando os pesos da melhor época — previne overfitting ao
impedir que o modelo continue otimizando a loss de treino às custas da generalização.

# <font color='green' style='font-size: 30px;'> Loop de Treinamento </font>
<hr style='border: 2px solid green;'>

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(cnn_baseline.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

cnn_baseline, historico_cnn = treinar_modelo(
    cnn_baseline, train_loader, val_loader, criterion, optimizer, scheduler,
    epochs=30, device=DEVICE, early_stopping_patience=5, nome_modelo="CNN Baseline"
)

# <font color='green' style='font-size: 30px;'> Acompanhamento do Treinamento </font>
<hr style='border: 2px solid green;'>

In [ ]:
plot_historico_treino(historico_cnn, titulo="CNN Baseline — Histórico de Treinamento")

# <font color='green' style='font-size: 30px;'> Avaliação </font>
<hr style='border: 2px solid green;'>

In [ ]:
(y_true_cnn_test, y_pred_cnn_test, y_proba_cnn_test), tempo_inf_cnn = avaliar_modelo(cnn_baseline, test_loader)
(y_true_cnn_val, y_pred_cnn_val, y_proba_cnn_val), _ = avaliar_modelo(cnn_baseline, val_loader)
(y_true_cnn_train, y_pred_cnn_train, y_proba_cnn_train), _ = avaliar_modelo(cnn_baseline, train_loader)

metricas_cnn = {
    "treino": calcular_metricas(y_true_cnn_train, y_pred_cnn_train, y_proba_cnn_train),
    "validacao": calcular_metricas(y_true_cnn_val, y_pred_cnn_val, y_proba_cnn_val),
    "teste": calcular_metricas(y_true_cnn_test, y_pred_cnn_test, y_proba_cnn_test),
}
tabela_metricas(metricas_cnn)

In [ ]:
plot_matriz_confusao(y_true_cnn_test, y_pred_cnn_test, CLASSES, titulo="CNN Baseline — Matriz de Confusão (Teste)")
plot_curva_roc(y_true_cnn_test, y_proba_cnn_test, CLASSES, titulo="CNN Baseline — Curva ROC (Teste)")

# <font color='green' style='font-size: 30px;'> O que a CNN Aprendeu? </font>
<hr style='border: 2px solid green;'>

In [ ]:
visualizar_filtros(cnn_baseline, "conv1", n_filtros=32)

imagem_exemplo, _ = dataset_teste_final[0]
visualizar_feature_maps(cnn_baseline, "conv1", imagem_exemplo, n_maps=16)
visualizar_feature_maps(cnn_baseline, "conv3", imagem_exemplo, n_maps=16)

**Interpretação esperada:** os filtros da primeira camada (`conv1`) tendem a se especializar em
padrões simples — bordas, contrastes de cor, gradientes de textura. Os feature maps das
camadas finais (`conv3`) já representam combinações mais abstratas desses padrões,
correlacionadas a estruturas maiores da cena (blocos de plantação, malha viária, corpos
d'água) — a rede caminha de "bordas" para "texturas" e daí para "objetos"/estruturas.

# <font color='red' style='font-size: 40px;'> Otimização de Hiperparâmetros com Optuna </font>
<hr style='border: 2px solid red;'>

## Estratégias de Busca

| Estratégia | Descrição |
|---|---|
| Grid Search | Testa exaustivamente todas as combinações de um grid fixo — caro e ineficiente em espaços grandes |
| Random Search | Amostra combinações aleatoriamente — mais eficiente que Grid Search em espaços de alta dimensão |
| Bayesian Optimization | Usa um modelo probabilístico (surrogate) para guiar a busca para regiões promissoras, com base nos resultados anteriores |
| TPE (Tree-structured Parzen Estimator) | Algoritmo Bayesiano usado por padrão no Optuna — modela separadamente a distribuição de hiperparâmetros para trials "bons" e "ruins" |
| Pruning | Interrompe trials pouco promissores antes de completarem todas as épocas, economizando tempo computacional |

**Escolha:** `Optuna` com `TPE` (padrão) e `MedianPruner`, por oferecer boa relação entre
qualidade da busca e custo computacional frente a Grid/Random Search.

In [ ]:
def objective(trial):
    set_seed(SEED)

    # Espaço de busca
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    dropout1 = trial.suggest_float("dropout1", 0.1, 0.4)
    dropout2 = trial.suggest_float("dropout2", 0.3, 0.6)
    n_filtros_base = trial.suggest_categorical("n_filtros_base", [16, 32])

    train_loader_opt = DataLoader(dataset_treino_final, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader_opt = DataLoader(dataset_val_final, batch_size=batch_size, shuffle=False, num_workers=2)

    modelo_trial = EuroSATCNN(num_classes=NUM_CLASSES, dropout1=dropout1, dropout2=dropout2).to(DEVICE)
    criterion_trial = nn.CrossEntropyLoss()
    optimizer_trial = torch.optim.AdamW(modelo_trial.parameters(), lr=lr, weight_decay=weight_decay)

    modelo_trial, historico_trial = treinar_modelo(
        modelo_trial, train_loader_opt, val_loader_opt, criterion_trial, optimizer_trial,
        scheduler=None, epochs=8, device=DEVICE, early_stopping_patience=3,
        nome_modelo=f"Trial {trial.number}"
    )

    return min(historico_trial["val_loss"])


study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED),
                             pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=20)

print("Melhores hiperparâmetros encontrados:")
print(study.best_params)

In [ ]:
plot_optimization_history(study).show()
plot_param_importances(study).show()

# <font color='green' style='font-size: 30px;'> Avaliação Final — CNN Base vs CNN Otimizada </font>
<hr style='border: 2px solid green;'>

In [ ]:
melhores_params = study.best_params

cnn_otimizada = EuroSATCNN(
    num_classes=NUM_CLASSES,
    dropout1=melhores_params["dropout1"],
    dropout2=melhores_params["dropout2"],
).to(DEVICE)

train_loader_final = DataLoader(dataset_treino_final, batch_size=melhores_params["batch_size"], shuffle=True, num_workers=2)
val_loader_final = DataLoader(dataset_val_final, batch_size=melhores_params["batch_size"], shuffle=False, num_workers=2)

criterion_otim = nn.CrossEntropyLoss()
optimizer_otim = torch.optim.AdamW(
    cnn_otimizada.parameters(), lr=melhores_params["lr"], weight_decay=melhores_params["weight_decay"]
)
scheduler_otim = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_otim, mode="min", factor=0.5, patience=2)

cnn_otimizada, historico_cnn_otim = treinar_modelo(
    cnn_otimizada, train_loader_final, val_loader_final, criterion_otim, optimizer_otim, scheduler_otim,
    epochs=30, device=DEVICE, early_stopping_patience=5, nome_modelo="CNN Otimizada"
)

(y_true_otim, y_pred_otim, y_proba_otim), tempo_inf_otim = avaliar_modelo(cnn_otimizada, test_loader)
metricas_cnn_otim_teste = calcular_metricas(y_true_otim, y_pred_otim, y_proba_otim)

print("CNN Base (teste):     ", metricas_cnn["teste"])
print("CNN Otimizada (teste):", metricas_cnn_otim_teste)

# <font color='red' style='font-size: 40px;'> Fine-Tuning de uma ResNet </font>
<hr style='border: 2px solid red;'>

**Transfer Learning** consiste em reaproveitar um modelo pré-treinado em uma grande base de
dados (ImageNet, ~1,2M imagens) como ponto de partida, em vez de aprender representações do
zero. A intuição é que as camadas iniciais de uma CNN aprendem padrões genéricos (bordas,
texturas, cores) que são úteis para praticamente qualquer tarefa de visão computacional — só as
camadas finais (mais específicas à tarefa original) precisam ser reaprendidas.

**Estratégia de congelamento:** congelamos os pesos do backbone convolucional (`layer1` a
`layer3`) e treinamos apenas `layer4` + a nova camada de classificação — um meio-termo entre
"congelar tudo" (rápido, mas menos adaptado ao domínio de satélite) e "fine-tuning completo"
(mais lento, maior risco de overfitting em um dataset de porte moderado como o EuroSAT).

In [ ]:
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

# Congela todas as camadas por padrão
for param in resnet.parameters():
    param.requires_grad = False

# Descongela apenas o último bloco convolucional (layer4)
for param in resnet.layer4.parameters():
    param.requires_grad = True

# Substitui a camada final para as 10 classes do EuroSAT
resnet.fc = nn.Linear(resnet.fc.in_features, NUM_CLASSES)

resnet = resnet.to(DEVICE)

total_params_resnet, trainable_params_resnet = contar_parametros(resnet)
print(f"Parâmetros totais: {total_params_resnet:,} | Treináveis: {trainable_params_resnet:,} "
      f"({trainable_params_resnet/total_params_resnet:.1%})")

In [ ]:
criterion_resnet = nn.CrossEntropyLoss()
# Apenas os parâmetros com requires_grad=True são otimizados
optimizer_resnet = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, resnet.parameters()), lr=1e-4, weight_decay=1e-4
)
scheduler_resnet = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_resnet, mode="min", factor=0.5, patience=2)

resnet, historico_resnet = treinar_modelo(
    resnet, train_loader, val_loader, criterion_resnet, optimizer_resnet, scheduler_resnet,
    epochs=15, device=DEVICE, early_stopping_patience=4, nome_modelo="ResNet50"
)

plot_historico_treino(historico_resnet, titulo="ResNet50 — Histórico de Treinamento")

In [ ]:
(y_true_resnet, y_pred_resnet, y_proba_resnet), tempo_inf_resnet = avaliar_modelo(resnet, test_loader)
metricas_resnet_teste = calcular_metricas(y_true_resnet, y_pred_resnet, y_proba_resnet)

print("CNN Otimizada (teste):", metricas_cnn_otim_teste)
print("ResNet50 (teste):     ", metricas_resnet_teste)

plot_matriz_confusao(y_true_resnet, y_pred_resnet, CLASSES, titulo="ResNet50 — Matriz de Confusão (Teste)")

# <font color='red' style='font-size: 40px;'> Fine-Tuning de um Vision Transformer </font>
<hr style='border: 2px solid red;'>

## Patch Embeddings

O Vision Transformer (ViT) divide a imagem em patches (ex.: 16×16 pixels), "achata" cada patch
em um vetor e projeta linearmente esse vetor em uma dimensão fixa $d$ — de forma análoga a
como palavras (tokens) são embutidas em um Transformer de linguagem. Um token especial
`[CLS]` é adicionado à sequência para agregar a informação global usada na classificação.

## Self-Attention

Cada patch "atende" a todos os outros patches da imagem, ponderando sua importância mútua:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V
$$

onde $Q$ (query), $K$ (key) e $V$ (value) são projeções lineares dos embeddings de entrada, e
$d_k$ é a dimensão das chaves (usada para normalizar a magnitude dos produtos internos).

## Diferenças Estruturais frente a CNNs

| CNN | ViT |
|---|---|
| Viés indutivo espacial forte (localidade, invariância translacional via pesos compartilhados) | Pouco viés indutivo — aprende relações espaciais a partir dos dados |
| Campo receptivo cresce gradualmente com a profundidade | Cada camada já enxerga a imagem inteira (atenção global) |
| Eficiente com datasets pequenos/médios | Tipicamente precisa de mais dados (ou pré-treino em larga escala) para performar bem |

In [ ]:
if timm is None:
    raise ImportError("Instale a biblioteca 'timm' para carregar o Vision Transformer: pip install timm")

vit = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=NUM_CLASSES)
vit = vit.to(DEVICE)

# Congela o backbone e treina apenas a cabeça de classificação + últimos blocos de atenção
for param in vit.parameters():
    param.requires_grad = False
for param in vit.head.parameters():
    param.requires_grad = True
for param in vit.blocks[-2:].parameters():
    param.requires_grad = True

total_params_vit, trainable_params_vit = contar_parametros(vit)
print(f"Parâmetros totais: {total_params_vit:,} | Treináveis: {trainable_params_vit:,} "
      f"({trainable_params_vit/total_params_vit:.1%})")

In [ ]:
criterion_vit = nn.CrossEntropyLoss()
optimizer_vit = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, vit.parameters()), lr=1e-4, weight_decay=1e-4
)
scheduler_vit = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_vit, mode="min", factor=0.5, patience=2)

vit, historico_vit = treinar_modelo(
    vit, train_loader, val_loader, criterion_vit, optimizer_vit, scheduler_vit,
    epochs=15, device=DEVICE, early_stopping_patience=4, nome_modelo="ViT"
)

plot_historico_treino(historico_vit, titulo="Vision Transformer — Histórico de Treinamento")

In [ ]:
(y_true_vit, y_pred_vit, y_proba_vit), tempo_inf_vit = avaliar_modelo(vit, test_loader)
metricas_vit_teste = calcular_metricas(y_true_vit, y_pred_vit, y_proba_vit)
print("ViT (teste):", metricas_vit_teste)

plot_matriz_confusao(y_true_vit, y_pred_vit, CLASSES, titulo="ViT — Matriz de Confusão (Teste)")

# <font color='red' style='font-size: 40px;'> Fine-Tuning utilizando LoRA </font>
<hr style='border: 2px solid red;'>

**Low-Rank Adaptation (LoRA)** parte da observação de que a atualização de pesos $\Delta W$
necessária para adaptar um modelo pré-treinado a uma nova tarefa costuma ter **baixo posto
(rank)**. Em vez de treinar a matriz de pesos completa $W \in \mathbb{R}^{d \times k}$, o LoRA
a mantém **congelada** e aprende apenas uma decomposição de baixo posto:

$$
W' = W + \Delta W = W + BA, \qquad B \in \mathbb{R}^{d \times r},\; A \in \mathbb{R}^{r \times k},\; r \ll \min(d,k)
$$

O hiperparâmetro `alpha` escala a contribuição de $BA$ (`scaling = alpha / r`). Como $r$ é
pequeno, o número de parâmetros treináveis cai drasticamente frente ao fine-tuning completo,
reduzindo custo de memória e tempo de treino — mantendo, na prática, performance próxima à do
fine-tuning completo em muitas tarefas.

Aplicamos LoRA nas projeções `qkv` (query/key/value) dos blocos de atenção do ViT.

In [ ]:
if LoraConfig is None or get_peft_model is None:
    raise ImportError("Instale a biblioteca 'peft' para usar LoRA: pip install peft")

vit_lora_base = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=NUM_CLASSES)

lora_config = LoraConfig(
    r=8,                       # rank da decomposição
    lora_alpha=16,             # fator de escala (scaling = alpha / r)
    target_modules=["qkv"],    # camadas de atenção adaptadas
    lora_dropout=0.1,
    bias="none",
)

vit_lora = get_peft_model(vit_lora_base, lora_config).to(DEVICE)
vit_lora.print_trainable_parameters()

In [ ]:
criterion_lora = nn.CrossEntropyLoss()
optimizer_lora = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, vit_lora.parameters()), lr=1e-3, weight_decay=1e-4
)
scheduler_lora = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_lora, mode="min", factor=0.5, patience=2)

vit_lora, historico_lora = treinar_modelo(
    vit_lora, train_loader, val_loader, criterion_lora, optimizer_lora, scheduler_lora,
    epochs=15, device=DEVICE, early_stopping_patience=4, nome_modelo="ViT + LoRA"
)

plot_historico_treino(historico_lora, titulo="ViT + LoRA — Histórico de Treinamento")

In [ ]:
(y_true_lora, y_pred_lora, y_proba_lora), tempo_inf_lora = avaliar_modelo(vit_lora, test_loader)
metricas_lora_teste = calcular_metricas(y_true_lora, y_pred_lora, y_proba_lora)

total_params_lora, trainable_params_lora = contar_parametros(vit_lora)

print("ViT Fine-Tuning Completo — treináveis:", f"{trainable_params_vit:,}", "| métricas teste:", metricas_vit_teste)
print("ViT + LoRA               — treináveis:", f"{trainable_params_lora:,}", "| métricas teste:", metricas_lora_teste)

# <font color='red' style='font-size: 40px;'> Extração de Embeddings </font>
<hr style='border: 2px solid red;'>

Um **embedding** é uma representação vetorial densa e de dimensão fixa que resume o conteúdo
semântico de uma imagem, extraída de uma camada intermediária (tipicamente a penúltima) da
rede — antes da camada de classificação final. Duas imagens semanticamente similares tendem a
gerar embeddings próximos no espaço vetorial (em distância euclidiana ou cosseno).

Selecionamos o backbone do **modelo com melhor F1-score no teste** (dentre CNN Otimizada,
ResNet50, ViT e ViT+LoRA — ver comparação consolidada na Seção 17) como extrator de
embeddings.

In [ ]:
# Seleciona automaticamente o melhor modelo com base no F1-score de teste calculado até aqui
candidatos = {
    "CNN Otimizada": (cnn_otimizada, metricas_cnn_otim_teste["f1"]),
    "ResNet50": (resnet, metricas_resnet_teste["f1"]),
    "ViT": (vit, metricas_vit_teste["f1"]),
    "ViT + LoRA": (vit_lora, metricas_lora_teste["f1"]),
}
nome_melhor_modelo = max(candidatos, key=lambda k: candidatos[k][1])
melhor_modelo = candidatos[nome_melhor_modelo][0]
print(f"Modelo selecionado para extração de embeddings: {nome_melhor_modelo}")

In [ ]:
# Remove a camada de classificação final para extrair o vetor de embedding (penúltima camada)
melhor_modelo_backbone = copy.deepcopy(melhor_modelo)

if hasattr(melhor_modelo_backbone, "fc"):
    melhor_modelo_backbone.fc = nn.Identity()
elif hasattr(melhor_modelo_backbone, "head"):
    melhor_modelo_backbone.head = nn.Identity()
elif hasattr(melhor_modelo_backbone, "classifier"):
    melhor_modelo_backbone.classifier = nn.Identity()

melhor_modelo_backbone = melhor_modelo_backbone.to(DEVICE)

embeddings_teste, labels_teste = extrair_embeddings(melhor_modelo_backbone, test_loader)
embeddings_treino, labels_treino = extrair_embeddings(melhor_modelo_backbone, train_loader)

print(f"Dimensão do embedding: {embeddings_teste.shape[1]}")
print(f"Nº de amostras — treino: {embeddings_treino.shape[0]} | teste: {embeddings_teste.shape[0]}")

# Persistência dos embeddings
os.makedirs("data_output", exist_ok=True)
np.save("data_output/embeddings_treino.npy", embeddings_treino)
np.save("data_output/embeddings_teste.npy", embeddings_teste)
np.save("data_output/labels_treino.npy", labels_treino)
np.save("data_output/labels_teste.npy", labels_teste)

# <font color='red' style='font-size: 40px;'> Classificação utilizando Machine Learning </font>
<hr style='border: 2px solid red;'>

Avaliamos se um modelo de Machine Learning **clássico** (LightGBM), treinado apenas sobre os
embeddings extraídos, consegue performance competitiva frente aos modelos neurais fim-a-fim —
um cenário comum em produção, onde reextrair embeddings uma única vez e treinar/atualizar um
modelo leve sobre eles é operacionalmente mais barato que retreinar a rede neural inteira.

In [ ]:
X_train_emb, X_val_emb, y_train_emb, y_val_emb = train_test_split(
    embeddings_treino, labels_treino, test_size=0.15, random_state=SEED, stratify=labels_treino
)

lgb_dataset_treino = lgb.Dataset(X_train_emb, label=y_train_emb)
lgb_dataset_val = lgb.Dataset(X_val_emb, label=y_val_emb, reference=lgb_dataset_treino)

params_lgb = {
    "objective": "multiclass",
    "num_class": NUM_CLASSES,
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "seed": SEED,
    "verbosity": -1,
}

inicio_lgb = time.time()
modelo_lgb = lgb.train(
    params_lgb, lgb_dataset_treino, num_boost_round=500,
    valid_sets=[lgb_dataset_val],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(period=50)]
)
tempo_treino_lgb = time.time() - inicio_lgb

In [ ]:
inicio_inf_lgb = time.time()
proba_lgb_teste = modelo_lgb.predict(embeddings_teste)
tempo_inf_lgb = time.time() - inicio_inf_lgb

pred_lgb_teste = proba_lgb_teste.argmax(axis=1)
metricas_lgb_teste = calcular_metricas(labels_teste, pred_lgb_teste, proba_lgb_teste)

print(f"LightGBM sobre embeddings ({nome_melhor_modelo}) — teste:", metricas_lgb_teste)
print(f"Comparação — {nome_melhor_modelo} fim-a-fim (teste):", candidatos[nome_melhor_modelo][1])

plot_matriz_confusao(labels_teste, pred_lgb_teste, CLASSES,
                      titulo=f"LightGBM sobre Embeddings ({nome_melhor_modelo}) — Matriz de Confusão")

# <font color='red' style='font-size: 40px;'> Visualização dos Embeddings </font>
<hr style='border: 2px solid red;'>

Reduzimos a dimensionalidade dos embeddings para 2D com **t-SNE** e **UMAP**, técnicas não
lineares de redução de dimensionalidade que preservam relações de vizinhança local. Se as
classes aparecerem bem separadas nesse espaço 2D, isso é um indício de que o backbone
aprendeu representações semanticamente ricas e discriminativas.

In [ ]:
amostra_idx = np.random.RandomState(SEED).choice(len(embeddings_teste), size=min(2000, len(embeddings_teste)), replace=False)
emb_amostra = embeddings_teste[amostra_idx]
labels_amostra = labels_teste[amostra_idx]

tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, init="pca")
emb_tsne = tsne.fit_transform(emb_amostra)

plt.figure(figsize=(9, 7))
sns.scatterplot(x=emb_tsne[:, 0], y=emb_tsne[:, 1],
                 hue=[CLASSES[l] for l in labels_amostra], palette="tab10", s=15, alpha=0.7)
plt.title(f"t-SNE dos Embeddings — {nome_melhor_modelo}")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
if umap is not None:
    reducer = umap.UMAP(n_components=2, random_state=SEED)
    emb_umap = reducer.fit_transform(emb_amostra)

    plt.figure(figsize=(9, 7))
    sns.scatterplot(x=emb_umap[:, 0], y=emb_umap[:, 1],
                     hue=[CLASSES[l] for l in labels_amostra], palette="tab10", s=15, alpha=0.7)
    plt.title(f"UMAP dos Embeddings — {nome_melhor_modelo}")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("Biblioteca 'umap-learn' não encontrada — instale com: pip install umap-learn")

**Discussão esperada:** classes com semântica visual próxima (ex.: `AnnualCrop`,
`PermanentCrop` e `Pasture` — todas texturas agrícolas) tendem a formar clusters mais próximos
ou parcialmente sobrepostos, enquanto classes visualmente distintas (ex.: `SeaLake`,
`Residential`) tendem a formar clusters bem isolados. Sobreposição entre clusters é um
indicador direto de onde o classificador tende a errar (correlaciona-se com a matriz de
confusão da Seção 7/9/10/11).

# <font color='red' style='font-size: 40px;'> Interpretabilidade </font>
<hr style='border: 2px solid red;'>

# <font color='green' style='font-size: 30px;'> Grad-CAM (CNN) </font>
<hr style='border: 2px solid green;'>

O **Grad-CAM** (Gradient-weighted Class Activation Mapping) usa os gradientes da classe-alvo
em relação aos feature maps da última camada convolucional para produzir um mapa de calor,
indicando quais regiões da imagem mais contribuíram para a decisão do modelo:

$$
\alpha_k^c = \frac{1}{Z}\sum_i \sum_j \frac{\partial y^c}{\partial A_{ij}^k}, \qquad
L_{Grad\text{-}CAM}^c = \text{ReLU}\left(\sum_k \alpha_k^c A^k\right)
$$

onde $A^k$ é o k-ésimo feature map da última camada convolucional e $y^c$ é o score (logit) da
classe $c$.

In [ ]:
def grad_cam(modelo, imagem_tensor, nome_camada_alvo, classe_alvo=None, device=DEVICE):
    """Calcula o mapa de calor Grad-CAM para uma imagem e camada convolucional-alvo."""
    modelo.eval()
    ativacoes, gradientes = {}, {}

    def forward_hook(module, entrada, saida):
        ativacoes["valor"] = saida

    def backward_hook(module, grad_entrada, grad_saida):
        gradientes["valor"] = grad_saida[0]

    camada = dict(modelo.named_modules())[nome_camada_alvo]
    h1 = camada.register_forward_hook(forward_hook)
    h2 = camada.register_full_backward_hook(backward_hook)

    entrada = imagem_tensor.unsqueeze(0).to(device)
    entrada.requires_grad_(True)

    saida = modelo(entrada)
    if classe_alvo is None:
        classe_alvo = saida.argmax(dim=1).item()

    modelo.zero_grad()
    saida[0, classe_alvo].backward()

    grads = gradientes["valor"][0]          # (C, H, W)
    ativ = ativacoes["valor"][0]              # (C, H, W)

    pesos = grads.mean(dim=(1, 2))            # média espacial -> (C,)
    cam = torch.relu((pesos[:, None, None] * ativ).sum(dim=0))
    cam = cam.detach().cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    h1.remove(); h2.remove()
    return cam, classe_alvo


def plot_grad_cam(imagem_tensor, cam, titulo="Grad-CAM"):
    img = imagem_tensor.permute(1, 2, 0).cpu().numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    cam_resized = cv2.resize(cam, (img.shape[1], img.shape[0]))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET) / 255.0
    overlay = 0.5 * img + 0.5 * heatmap[..., ::-1]

    fig, axes = plt.subplots(1, 3, figsize=(11, 4))
    axes[0].imshow(img); axes[0].set_title("Imagem Original"); axes[0].axis("off")
    axes[1].imshow(cam_resized, cmap="jet"); axes[1].set_title("Mapa de Ativação"); axes[1].axis("off")
    axes[2].imshow(overlay); axes[2].set_title("Sobreposição"); axes[2].axis("off")
    plt.suptitle(titulo)
    plt.tight_layout()
    plt.show()


imagem_gradcam, label_gradcam = dataset_teste_final[0]
cam, classe_predita = grad_cam(cnn_otimizada, imagem_gradcam, nome_camada_alvo="conv3")
plot_grad_cam(imagem_gradcam, cam,
              titulo=f"Grad-CAM (CNN) — Real: {CLASSES[label_gradcam]} | Predito: {CLASSES[classe_predita]}")

# <font color='green' style='font-size: 30px;'> Attention Maps (ViT) </font>
<hr style='border: 2px solid green;'>

No ViT, a interpretabilidade é obtida diretamente dos **pesos de atenção** do token `[CLS]`
em relação a todos os patches da imagem na última camada — indicando quais regiões o modelo
"olhou" para formar sua representação de classificação.

In [ ]:
def attention_map_vit(modelo_vit, imagem_tensor, device=DEVICE):
    """Extrai o mapa de atenção do token [CLS] na última camada de atenção do ViT."""
    modelo_vit.eval()
    atencoes = {}

    def hook(module, entrada, saida):
        atencoes["valor"] = saida

    # Compatível com a implementação timm (blocks[-1].attn)
    ultimo_bloco_attn = modelo_vit.blocks[-1].attn if hasattr(modelo_vit, "blocks") else modelo_vit.base_model.blocks[-1].attn
    handle = ultimo_bloco_attn.register_forward_hook(hook)

    entrada = imagem_tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        modelo_vit(entrada)
    handle.remove()

    # Fallback simples caso a versão do timm não exponha os pesos de atenção diretamente:
    # usa-se a norma da ativação de saída como proxy de relevância por patch.
    saida_attn = atencoes["valor"][0]
    n_patches = saida_attn.shape[0] - 1  # exclui o token [CLS]
    lado = int(np.sqrt(n_patches))
    relevancia = saida_attn[1:].norm(dim=-1).reshape(lado, lado).detach().cpu().numpy()
    relevancia = (relevancia - relevancia.min()) / (relevancia.max() - relevancia.min() + 1e-8)
    return relevancia


imagem_attn, label_attn = dataset_teste_final[1]
mapa_attn = attention_map_vit(vit, imagem_attn)
plot_grad_cam(imagem_attn, mapa_attn, titulo=f"Attention Map (ViT) — Real: {CLASSES[label_attn]}")

**Comparação visual:** espera-se que o Grad-CAM da CNN destaque regiões mais localizadas e
compactas (coerente com o viés indutivo espacial das convoluções), enquanto o mapa de atenção
do ViT tende a distribuir relevância de forma mais ampla pela imagem, refletindo seu mecanismo
de atenção global desde as primeiras camadas.

# <font color='red' style='font-size: 40px;'> Robustez dos Modelos </font>
<hr style='border: 2px solid red;'>

Avaliamos a degradação de performance de cada modelo sob perturbações que simulam condições
adversas de captura (nuvens/névoa, ruído do sensor, baixa resolução do satélite, variação de
iluminação/estação do ano) — relevante para avaliar a confiabilidade dos modelos em produção,
fora das condições "limpas" do conjunto de teste original.

In [ ]:
def aplicar_blur(img_tensor, kernel_size=5):
    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_blur = cv2.GaussianBlur(img_np, (kernel_size, kernel_size), 0)
    return torch.from_numpy(img_blur).permute(2, 0, 1)


def aplicar_ruido_gaussiano(img_tensor, std=0.1):
    ruido = torch.randn_like(img_tensor) * std
    return img_tensor + ruido


def aplicar_baixa_resolucao(img_tensor, fator=4):
    c, h, w = img_tensor.shape
    img_reduzida = F.interpolate(img_tensor.unsqueeze(0), size=(h // fator, w // fator), mode="bilinear")
    img_restaurada = F.interpolate(img_reduzida, size=(h, w), mode="bilinear")
    return img_restaurada.squeeze(0)


def aplicar_variacao_iluminacao(img_tensor, fator_brilho=1.5):
    return torch.clamp(img_tensor * fator_brilho, img_tensor.min(), img_tensor.max())


PERTURBACOES = {
    "Original": lambda x: x,
    "Blur": aplicar_blur,
    "Ruído Gaussiano": aplicar_ruido_gaussiano,
    "Baixa Resolução": aplicar_baixa_resolucao,
    "Variação de Iluminação": aplicar_variacao_iluminacao,
}


def avaliar_robustez(modelo, dataset, perturbacoes, device=DEVICE, n_amostras=500):
    """Avalia a Accuracy de um modelo sob diferentes perturbações do conjunto de teste."""
    indices = np.random.RandomState(SEED).choice(len(dataset), size=min(n_amostras, len(dataset)), replace=False)
    resultados = {}

    for nome_pert, func_pert in perturbacoes.items():
        y_true, y_pred = [], []
        modelo.eval()
        with torch.no_grad():
            for idx in indices:
                img, label = dataset[idx]
                img_pert = func_pert(img).unsqueeze(0).to(device)
                saida = modelo(img_pert)
                y_pred.append(saida.argmax(1).item())
                y_true.append(label)
        resultados[nome_pert] = accuracy_score(y_true, y_pred)

    return resultados


modelos_robustez = {
    "CNN Otimizada": cnn_otimizada,
    "ResNet50": resnet,
    "ViT": vit,
    "ViT + LoRA": vit_lora,
}

resultados_robustez = {
    nome: avaliar_robustez(modelo, dataset_teste_final, PERTURBACOES)
    for nome, modelo in modelos_robustez.items()
}

df_robustez = pd.DataFrame(resultados_robustez).T
df_robustez

In [ ]:
df_robustez.plot(kind="bar", figsize=(11, 5))
plt.title("Robustez a Perturbações — Accuracy por Modelo e Tipo de Perturbação")
plt.ylabel("Accuracy")
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# <font color='red' style='font-size: 40px;'> Comparação dos Modelos </font>
<hr style='border: 2px solid red;'>

In [ ]:
resultados_finais = [
    {
        "modelo": "CNN Base",
        **metricas_cnn["teste"],
        "tempo_inferencia_s": tempo_inf_cnn,
        "parametros_totais": contar_parametros(cnn_baseline)[0],
        "parametros_treinaveis": contar_parametros(cnn_baseline)[1],
        "tamanho_mb": tamanho_modelo_mb(cnn_baseline),
    },
    {
        "modelo": "CNN Otimizada (Optuna)",
        **metricas_cnn_otim_teste,
        "tempo_inferencia_s": tempo_inf_otim,
        "parametros_totais": contar_parametros(cnn_otimizada)[0],
        "parametros_treinaveis": contar_parametros(cnn_otimizada)[1],
        "tamanho_mb": tamanho_modelo_mb(cnn_otimizada),
    },
    {
        "modelo": "ResNet50 (Transfer Learning)",
        **metricas_resnet_teste,
        "tempo_inferencia_s": tempo_inf_resnet,
        "parametros_totais": contar_parametros(resnet)[0],
        "parametros_treinaveis": contar_parametros(resnet)[1],
        "tamanho_mb": tamanho_modelo_mb(resnet),
    },
    {
        "modelo": "ViT (Fine-Tuning)",
        **metricas_vit_teste,
        "tempo_inferencia_s": tempo_inf_vit,
        "parametros_totais": contar_parametros(vit)[0],
        "parametros_treinaveis": contar_parametros(vit)[1],
        "tamanho_mb": tamanho_modelo_mb(vit),
    },
    {
        "modelo": "ViT + LoRA",
        **metricas_lora_teste,
        "tempo_inferencia_s": tempo_inf_lora,
        "parametros_totais": total_params_lora,
        "parametros_treinaveis": trainable_params_lora,
        "tamanho_mb": tamanho_modelo_mb(vit_lora),
    },
    {
        "modelo": f"Embeddings ({nome_melhor_modelo}) + LightGBM",
        **metricas_lgb_teste,
        "tempo_treino_s": tempo_treino_lgb,
        "tempo_inferencia_s": tempo_inf_lgb,
        "parametros_totais": np.nan,
        "parametros_treinaveis": np.nan,
        "tamanho_mb": os.path.getsize("data_output/embeddings_treino.npy") / (1024**2) if os.path.exists("data_output/embeddings_treino.npy") else np.nan,
    },
]

df_comparacao = comparar_modelos(resultados_finais)
df_comparacao

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_comparacao.plot(x="modelo", y=["accuracy", "f1", "auc"], kind="bar", ax=axes[0])
axes[0].set_title("Métricas Preditivas por Modelo")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend(loc="lower right")

df_comparacao.plot(x="modelo", y="tempo_inferencia_s", kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Tempo de Inferência por Modelo (s)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# <font color='red' style='font-size: 40px;'> Trade-offs </font>
<hr style='border: 2px solid red;'>

Com base na tabela consolidada da Seção 17 e nos resultados de robustez da Seção 16,
discutimos os trade-offs entre as estratégias avaliadas:

- **Desempenho:** modelos pré-treinados (ResNet50, ViT) tendem a superar a CNN treinada do
  zero em Accuracy/F1, especialmente com dataset de porte moderado — o pré-treino em ImageNet
  fornece representações visuais genéricas difíceis de igualar treinando do zero.
- **Custo computacional / velocidade de treino:** a CNN baseline é a mais barata para treinar
  do zero; o ViT com fine-tuning completo é o mais custoso; o **LoRA reduz drasticamente** o
  custo de treino do ViT sem abrir mão de grande parte da performance.
- **Velocidade de inferência:** arquiteturas menores (CNN) tendem a ser mais rápidas em
  inferência que ViT/ResNet50 — relevante para cenários de baixa latência.
- **Interpretabilidade:** Grad-CAM (CNN/ResNet) produz mapas mais localizados e intuitivos que
  os Attention Maps do ViT, que tendem a ser mais difusos — pode pesar em cenários com exigência
  de explicabilidade regulatória.
- **Robustez:** conforme a Seção 16, espera-se que modelos com maior capacidade
  representacional (ViT, ResNet) degradem menos sob perturbações do que a CNN rasa, mas isso
  deve ser confirmado empiricamente nos resultados obtidos.
- **Escalabilidade e facilidade de manutenção:** a abordagem de **Embeddings + LightGBM**
  se destaca pela facilidade de retreinar/atualizar o classificador (segundos, não minutos)
  sem tocar na rede neural — interessante quando o backbone é estável e apenas a fronteira de
  decisão precisa ser ajustada com frequência.
- **Capacidade de generalização:** a comparação treino × validação × teste (Seções 7, 9, 10 e
  11) deve ser revisada para garantir que nenhum modelo apresente gap elevado entre treino e
  teste (sinal de overfitting) — o que violaria o princípio de simplicidade adotado neste
  projeto (Regra 11 da especificação).

### Quando cada estratégia é mais indicada

| Cenário | Estratégia recomendada |
|---|---|
| Poucos dados rotulados, sem GPU robusta | Embeddings pré-treinados + LightGBM |
| Necessidade de interpretabilidade regulatória forte | CNN / ResNet + Grad-CAM |
| Restrição de latência de inferência em produção | CNN baseline (ou ResNet com poda/quantização) |
| Orçamento de treino limitado, mas ganho de performance desejado | ViT + LoRA |
| Máxima performance, orçamento de treino disponível | ViT com fine-tuning completo (ou ResNet completo) |

# <font color='red' style='font-size: 40px;'> Engenharia de Atributos baseada em Embeddings </font>
<hr style='border: 2px solid red;'>

A partir dos embeddings extraídos (Seção 12), construímos indicadores derivados úteis tanto
como features adicionais para modelos downstream quanto como sinais de monitoramento em
produção (ex.: detecção de casos de baixa confiança ou de drift de distribuição).

In [ ]:
def norma_l2(embeddings):
    """Norma L2 de cada vetor de embedding — magnitude geral da representação."""
    return np.linalg.norm(embeddings, axis=1)


def similaridade_cosseno_media(embeddings, k=5):
    """Similaridade de cosseno média de cada ponto em relação aos k vizinhos mais próximos."""
    from sklearn.metrics.pairwise import cosine_similarity
    sim_matrix = cosine_similarity(embeddings)
    np.fill_diagonal(sim_matrix, -np.inf)
    top_k = np.sort(sim_matrix, axis=1)[:, -k:]
    return top_k.mean(axis=1)


def distancia_ao_centroide(embeddings, labels):
    """Distância euclidiana de cada embedding ao centróide da sua própria classe."""
    distancias = np.zeros(len(embeddings))
    for classe in np.unique(labels):
        idx_classe = labels == classe
        centroide = embeddings[idx_classe].mean(axis=0)
        distancias[idx_classe] = np.linalg.norm(embeddings[idx_classe] - centroide, axis=1)
    return distancias


def score_confianca_e_entropia_margem(proba):
    """
    A partir das probabilidades preditas, calcula:
    - score de confiança (probabilidade da classe mais provável)
    - entropia da distribuição de probabilidades
    - margem entre a 1ª e a 2ª classe mais prováveis
    """
    proba_ordenada = np.sort(proba, axis=1)[:, ::-1]
    confianca = proba_ordenada[:, 0]
    margem = proba_ordenada[:, 0] - proba_ordenada[:, 1]
    entropia = -np.sum(proba * np.log(proba + 1e-12), axis=1)
    return confianca, entropia, margem


df_features_embeddings = pd.DataFrame({
    "norma_l2": norma_l2(embeddings_teste),
    "similaridade_media_k5": similaridade_cosseno_media(embeddings_teste),
    "distancia_centroide": distancia_ao_centroide(embeddings_teste, labels_teste),
})

confianca, entropia, margem = score_confianca_e_entropia_margem(proba_lgb_teste)
df_features_embeddings["confianca"] = confianca
df_features_embeddings["entropia"] = entropia
df_features_embeddings["margem"] = margem
df_features_embeddings["classe"] = [CLASSES[l] for l in labels_teste]

df_features_embeddings.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(df_features_embeddings["confianca"], ax=axes[0], bins=30)
axes[0].set_title("Distribuição do Score de Confiança")

sns.scatterplot(data=df_features_embeddings, x="distancia_centroide", y="confianca",
                 hue="classe", palette="tab10", s=15, alpha=0.6, ax=axes[1], legend=False)
axes[1].set_title("Distância ao Centróide vs. Confiança da Predição")
plt.tight_layout()
plt.show()

**Aplicações práticas:** amostras com **baixa confiança + alta entropia + alta distância ao
centróide** são boas candidatas para revisão manual (fila de amostragem ativa) ou para
sinalizar **drift** quando aparecem em maior proporção em produção do que no conjunto de
teste original — um sinal de que a distribuição de entrada mudou e o modelo pode precisar de
retreinamento.

# <font color='red' style='font-size: 40px;'> Conclusões </font>
<hr style='border: 2px solid red;'>

> **Nota:** as respostas abaixo devem ser revisadas e ajustadas após a execução completa do
> notebook com os dados reais, substituindo as afirmações condicionais por conclusões
> definitivas baseadas na tabela da Seção 17 e nos gráficos de robustez da Seção 16.

**Qual estratégia apresentou a melhor performance?**
A ser respondido com base na linha de maior F1-score/AUC em `df_comparacao` (Seção 17) —
tipicamente espera-se que ResNet50 ou ViT (fine-tuning completo) liderem em métricas puras de
performance, dado o pré-treino em ImageNet.

**Qual apresentou o melhor custo-benefício?**
Tipicamente **ViT + LoRA** ou **Embeddings + LightGBM**, por entregarem performance próxima
aos modelos mais pesados a uma fração do custo computacional de treino/manutenção.

**Quando vale a pena treinar uma CNN do zero?**
Quando não há modelo pré-treinado adequado ao domínio (ex.: imagens muito distintas de fotos
naturais, como certos sensores multiespectrais), ou quando restrições de tamanho/latência do
modelo em produção são mais importantes que ganhos marginais de acurácia.

**Quando utilizar Transfer Learning?**
Quando o domínio da tarefa tem alguma proximidade visual com o dataset de pré-treino
(ImageNet) e há volume de dados rotulados moderado — como é o caso de imagens de satélite RGB.

**Quando utilizar Fine-Tuning completo?**
Quando há orçamento computacional disponível, volume de dados suficiente para evitar
overfitting em um modelo com muitos parâmetros treináveis, e a performance marginal adicional
justifica o custo.

**Quando utilizar LoRA?**
Quando se deseja adaptar um modelo grande pré-treinado a um novo domínio com recursos
computacionais limitados (memória/tempo), ou quando é necessário manter múltiplas adaptações
leves de um mesmo backbone (ex.: um adaptador LoRA por tipo de sensor/satélite).

**Em quais cenários embeddings + ML clássico são suficientes?**
Quando o backbone já captura bem a semântica do domínio e o que resta é apenas ajustar a
fronteira de decisão — cenário comum quando o conjunto de classes muda com frequência, mas o
domínio visual (satélite) permanece o mesmo.

**Quanto a otimização de hiperparâmetros melhorou a CNN?**
Comparar `metricas_cnn["teste"]` vs. `metricas_cnn_otim_teste` (Seção 8) — reportar o ganho
percentual absoluto em F1-score e Accuracy.

**Quais limitações foram encontradas?**
- Dependência de GPU para viabilizar o treino de ResNet50/ViT em tempo hábil.
- Tamanho do EuroSAT (relativamente pequeno) limita o ganho potencial de arquiteturas muito
  grandes, que tendem a performar melhor com datasets massivos.
- O proxy de Attention Map utilizado (norma da ativação) é uma aproximação simplificada — uma
  extração direta dos pesos de atenção (`attn_weights`) tende a ser mais fiel, a depender da
  versão da biblioteca `timm`/`transformers` utilizada.

**Quais são os próximos passos / trabalhos futuros?**
- Testar quantização e poda (pruning) dos modelos vencedores para reduzir custo de deploy.
- Avaliar arquiteturas híbridas CNN+Transformer (ex.: ConvNeXt, Swin Transformer).
- Expandir a análise de robustez para cenários reais de degradação de imagens de satélite
  (nuvens, sombras, artefatos de compressão).
- Investigar detecção de drift em produção usando os indicadores da Seção 19 como
  monitoramento contínuo.